# Key Metrics 

## Efficiency

**What is it?**<br/>
- using least amount of energy as possible to accomplish a task

**Why do we want to measure it?**
- as you climb more efficiently, you can climb with less energy expelled and allows you to climb higher grades
- can mitigate injuries as you aren't putting your body in compromised positions

**How can we measure it?** <br/>
- time spent with biceps in flexion
    - exponential dist? some flexion not punished but more you do more is punished
    - output with sigmoid to be b/w 0 and 1
- if both arms close to body and below head
- if time taken on climb is too long (possibly didnt read route)
    - could also be long route

## Mobility
- angle between femurs and center line

In [16]:
# %pip install mediapipe

In [17]:
import cv2
import time
import math
import numpy as np
import mediapipe as mp

# Following https://learnopencv.com/building-a-body-posture-analysis-system-using-mediapipe/

**Helper functions**

In [18]:
def distance(x1: float, y1: float, x2: float, y2: float) -> float:
    """
    uses the euclidean distance formula to calculate the distance b/w two points (x1, y1) and (x2, y2)
    
    Args:
        x1 (float): x-value of point 1
        y1 (float): y-value of point 1
        x2 (float): x-value of point 2
        y2 (float): y-value of point 2

    Returns:
        float: distance between two points
    """
    dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    return dist

In [19]:
def angle_between(x1: float, y1: float, x2: float, y2: float) -> float:
    """
    calculates the inner angle between two vectors: P_12 and P_13

    Args:
        x1 (float): x-value of point 1
        y1 (float): y-value of point 1
        x2 (float): x-value of point 2
        y2 (float): y-value of point 2

    Returns:
        float: theta in degrees
    """
    vec1 = np.array([x1, y1])
    vec2 = np.array([x2, y2])
    
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    cos_theta = np.dot(vec1, vec2) / (norm1 * norm2)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    
    theta_rads = np.arccos(cos_theta)
    theta_deg = np.degrees(theta_rads)
    
    return theta_deg

In [20]:
def put_text_with_background(image, text, position, font, scale, color, thickness, bg_color):
    (text_width, text_height), baseline = cv2.getTextSize(text, font, scale, thickness)
    bottom_left = (position[0], position[1] + baseline)
    top_right = (position[0] + text_width, position[1] - text_height - baseline)
    cv2.rectangle(image, bottom_left, top_right, bg_color, cv2.FILLED)
    cv2.putText(image, text, position, font, scale, color, thickness)

## Initializations Hyper Params

In [21]:
good_frames = 0
bad_frames = 0
font = cv2.FONT_HERSHEY_SIMPLEX
blue = (255, 127, 0)
red = (50, 50, 255)
green = (127, 255, 0)
dark_blue = (127, 20, 0)
light_green = (127, 233, 100)
yellow = (0, 255, 255)
pink = (255, 0, 255)
black = (0, 0, 0)

# initialize mediapipe pose class
# mp_pose = mp.solutions.pose
# pose = mp_pose.Pose()

**Create video capture and writer objects**

In [22]:
# file_name = 'tomoa.mp4'
# output = 'tomoa_tracked.mp4'
# cap = cv2.VideoCapture(file_name)

# # Meta
# fps = int(cap.get(cv2.CAP_PROP_FPS))
# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# frame_size = (width, height)
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# video_output = cv2.VideoWriter('test_out/output.mp4', fourcc, fps, frame_size)

**main loop**<br/>
https://docs.opencv.org/4.x/dd/d43/tutorial_py_video_display.html

- change color based on angle
- change metric name to climbing efficiency

In [23]:
import cv2
import mediapipe as mp
import numpy as np
import time


input_path = '../docs/tomoa_outside.mp4'
output_path = 'test_out/output_video2.mp4'

# Capture video
cap = cv2.VideoCapture(input_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

pTime = 0
min_angle = float('-inf')
good_frames = 0
bad_frames = 0
angle_threshold = 90

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height)) 

if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()
    
mpPose = mp.solutions.pose
pose = mpPose.Pose()
mpDraw = mp.solutions.drawing_utils

while True:
    success, img = cap.read()
    if not success:
        break

    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(imgRGB)
    
    if results.pose_landmarks:
        # mpDraw.draw_landmarks(img, 
        #                       results.pose_landmarks, 
        #                       mpPose.POSE_CONNECTIONS, 
        #                       mpDraw.DrawingSpec(color=yellow, thickness=2, circle_radius=2),
        #                       mpDraw.DrawingSpec(color=green, thickness=2, circle_radius=2))
        h, w, c = img.shape
        
        landmarks = results.pose_landmarks.landmark
        
        # Right arm landmarks
        r_shoulder = (landmarks[mpPose.PoseLandmark.RIGHT_SHOULDER.value].x * w, 
                      landmarks[mpPose.PoseLandmark.RIGHT_SHOULDER.value].y * h)
        r_elbow = (landmarks[mpPose.PoseLandmark.RIGHT_ELBOW.value].x * w, 
                   landmarks[mpPose.PoseLandmark.RIGHT_ELBOW.value].y * h)
        r_wrist = (landmarks[mpPose.PoseLandmark.RIGHT_WRIST.value].x * w, 
                   landmarks[mpPose.PoseLandmark.RIGHT_WRIST.value].y * h)
        
        # Left arm landmarks
        l_shoulder = (landmarks[mpPose.PoseLandmark.LEFT_SHOULDER.value].x * w, 
                      landmarks[mpPose.PoseLandmark.LEFT_SHOULDER.value].y * h)
        l_elbow = (landmarks[mpPose.PoseLandmark.LEFT_ELBOW.value].x * w, 
                   landmarks[mpPose.PoseLandmark.LEFT_ELBOW.value].y * h)
        l_wrist = (landmarks[mpPose.PoseLandmark.LEFT_WRIST.value].x * w, 
                   landmarks[mpPose.PoseLandmark.LEFT_WRIST.value].y * h)
        
        # arm angle calcs
        r_bicep_angle = angle_between(r_shoulder[0] - r_elbow[0], r_shoulder[1] - r_elbow[1], 
                                    r_wrist[0] - r_elbow[0], r_wrist[1] - r_elbow[1])

        l_bicep_angle = angle_between(l_shoulder[0] - l_elbow[0], l_shoulder[1] - l_elbow[1], 
                                    l_wrist[0] - l_elbow[0], l_wrist[1] - l_elbow[1])

        # cv2.putText(img, f"Left angle: {int(l_bicep_angle)}", (10, 100), cv2.FONT_HERSHEY_COMPLEX, .5, blue, 2)
        
        if int(r_bicep_angle) <= angle_threshold and int(l_bicep_angle) <= angle_threshold:
            bad_frames += 1
            mpDraw.draw_landmarks(img, 
                              results.pose_landmarks, 
                              mpPose.POSE_CONNECTIONS, 
                              mpDraw.DrawingSpec(color=yellow, thickness=2, circle_radius=2),
                              mpDraw.DrawingSpec(color=red, thickness=2, circle_radius=2))
        else:
            good_frames += 1
            mpDraw.draw_landmarks(img, 
                              results.pose_landmarks, 
                              mpPose.POSE_CONNECTIONS, 
                              mpDraw.DrawingSpec(color=yellow, thickness=2, circle_radius=2),
                              mpDraw.DrawingSpec(color=green, thickness=2, circle_radius=2))

        total_frames = good_frames + bad_frames
        frac_good = good_frames / total_frames

        put_text_with_background(image=img, 
                                 text=f"Climbing Efficiency: {(frac_good * 100.0):.2f}", 
                                 position=(10, 45), 
                                 font=cv2.FONT_HERSHEY_SIMPLEX, 
                                 scale=1, 
                                 color=blue, 
                                 thickness=1, 
                                 bg_color=black)
        # Count Good and Bad Framesq
        # put_text_with_background(img, f"Good: {good_frames}", (10,70), font, .5, green, 1, bg_color)
        # put_text_with_background(img, f"Bad: {bad_frames}", (10,100), font, .5, red, 1, bg_color)
        
    out.write(img)
    cv2.imshow("Image", img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
print(f"Bad Frames: {bad_frames}\nGood Frames: {good_frames}")
cap.release()
out.release()
cv2.destroyAllWindows()


Bad Frames: 80
Good Frames: 1084


In [ ]:
# test

In [24]:
# # if not cap.isOpened():
# #     print("Cannot open camera")
# #     exit()
    
# while cap.isOpened(): 
#     success, image = cap.read()
#     if not success:
#         print(f"Null.Frames")
#         break

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     h, w = image.shape[:2]

#     image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
#     keypoints = pose.process(image)
    
#     image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
#     lm = keypoints.pose_landmarks
#     lmPose = mp_pose.PoseLandmark

#     # nose
#     nose_x = int(lm.landmark[lmPose.NOSE].x * w)
#     nose_y = int(lm.landmark[lmPose.NOSE].y * h)
    # # left shoulder
    # l_shldr_x = int(lm.landmark[lmPose.LEFT_SHOULDER].x * w)
    # l_shldr_y = int(lm.landmark[lmPose.LEFT_SHOULDER].y * h)
    # # right shoulder
    # r_shldr_x = int(lm.landmark[lmPose.RIGHT_SHOULDER].x * w)
    # r_shldr_y = int(lm.landmark[lmPose.RIGHT_SHOULDER].y * h)
    # # left elbow
    # l_elbow_x = int(lm.landmark[lmPose.LEFT_ELBOW].x * w)
    # l_elbow_y = int(lm.landmark[lmPose.LEFT_ELBOW].y * h)
    # # right elbow
    # r_elbow_x = int(lm.landmark[lmPose.RIGHT_ELBOW].x * w)
    # r_elbow_y = int(lm.landmark[lmPose.RIGHT_ELBOW].y * h)
    # # left wrist
    # l_wrist_x = int(lm.landmark[lmPose.LEFT_WRIST].x * w)
    # l_wrist_y = int(lm.landmark[lmPose.LEFT_WRIST].y * h)
    # # right wrist
    # r_wrist_x = int(lm.landmark[lmPose.RIGHT_WRIST].x * w)
    # r_wrist_y = int(lm.landmark[lmPose.RIGHT_WRIST].y * h)
#     # left hip
#     l_hip_x = int(lm.landmark[lmPose.LEFT_HIP].x * w)
#     l_hip_y = int(lm.landmark[lmPose.LEFT_HIP].y * h)
#     # right hip
#     r_hip_x = int(lm.landmark[lmPose.RIGHT_HIP].x * w)
#     r_hip_y = int(lm.landmark[lmPose.RIGHT_HIP].y * h)
#     # left knee
#     l_knee_x = int(lm.landmark[lmPose.LEFT_KNEE].x * w)
#     l_knee_y = int(lm.landmark[lmPose.LEFT_KNEE].y * h)
#     # right knee
#     r_knee_x = int(lm.landmark[lmPose.RIGHT_KNEE].x * w)
#     r_knee_y = int(lm.landmark[lmPose.RIGHT_KNEE].y * h)
#     # left ankle
#     l_ankle_x = int(lm.landmark[lmPose.LEFT_ANKLE].x * w)
#     l_ankle_y = int(lm.landmark[lmPose.LEFT_ANKLE].y * h)
#     # right ankle
#     r_ankle_x = int(lm.landmark[lmPose.RIGHT_ANKLE].x * w)
#     r_ankle_y = int(lm.landmark[lmPose.RIGHT_ANKLE].y * h)
    
#     # Calculate angles
#     r_bicep_angle = angle_between(r_wrist_x, r_wrist_y, r_shldr_x, r_shldr_y)
#     l_bicep_angle = angle_between(l_wrist_x, l_wrist_y, l_shldr_x, l_shldr_y)

#     # Draw Circles
#     cv2.circle(image, (nose_x, nose_y), 7, yellow, -1)
    
#     cv2.circle(image, (r_shldr_x, r_shldr_y), 7, yellow, -1)
#     cv2.circle(image, (r_elbow_x, r_elbow_y), 7, yellow, -1)
#     cv2.circle(image, (r_wrist_x, r_wrist_y), 7, yellow, -1)
#     cv2.circle(image, (r_hip_x, r_hip_y), 7, yellow, -1)
#     cv2.circle(image, (r_knee_x, r_knee_y), 7, yellow, -1)
#     cv2.circle(image, (r_ankle_x, r_ankle_y), 7, yellow, -1)
    
#     cv2.circle(image, (l_shldr_x, l_shldr_y), 7, yellow, -1)
#     cv2.circle(image, (l_elbow_x, l_elbow_y), 7, yellow, -1)
#     cv2.circle(image, (l_wrist_x, l_wrist_y), 7, yellow, -1)
#     cv2.circle(image, (l_hip_x, l_hip_y), 7, yellow, -1)
#     cv2.circle(image, (l_knee_x, l_knee_y), 7, yellow, -1)
#     cv2.circle(image, (l_ankle_x, l_ankle_y), 7, yellow, -1)
    
#     # Draw Lines
#     cv2.line(image, (l_shldr_x, l_shldr_y), (r_shldr_x, r_shldr_y), green, 3)
#     cv2.line(image, (l_hip_x, l_hip_y), (r_hip_x, r_hip_y), green, 3)
    
#     cv2.line(image, (r_shldr_x, r_shldr_y), (r_elbow_x, r_elbow_y), green, 3)
#     cv2.line(image, (r_shldr_x, r_shldr_y), (r_hip_x, r_hip_y), green, 3)
#     cv2.line(image, (r_elbow_x, r_elbow_y),  (r_wrist_x, r_wrist_y), green, 3)
#     cv2.line(image, (r_hip_x, r_hip_y), (r_knee_x, r_knee_y), green, 3)
#     cv2.line(image, (r_knee_x, r_knee_y), (r_ankle_x, r_ankle_y), green, 3)
    
#     cv2.line(image, (l_shldr_x, l_shldr_y), (l_elbow_x, l_elbow_y), green, 3)
#     cv2.line(image, (l_shldr_x, l_shldr_y), (l_hip_x, l_hip_y), green, 3)
#     cv2.line(image, (l_elbow_x, l_elbow_y),  (l_wrist_x, l_wrist_y), green, 3)
#     cv2.line(image, (l_hip_x, l_hip_y), (l_knee_x, l_knee_y), green, 3)
#     cv2.line(image, (l_knee_x, l_knee_y), (l_ankle_x, l_ankle_y), green, 3)

#     angle_text_r = f"Right Bicep Angle: {str(round(r_bicep_angle, 2))}\nLeft Bicep Angle: {str(round(l_bicep_angle, 2))}"

#     cv2.imshow('frame', image)
#     if cv2.waitKey(1) & 0XFF == ord('q'):
#         break

# # When everything done, release the capture
# cap.release()
# cv2.destroyAllWindows()

In [25]:
# mpPose = mp.solutions.pose
# pose = mpPose.Pose()
# mpDraw = mp.solutions.drawing_utils

# #cap = cv2.VideoCapture(0)
# cap = cv2.VideoCapture('../docs/tomoa.mp4')
# pTime = 0

# while True:
#     success, img = cap.read()
#     imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     results = pose.process(imgRGB)
#     if results.pose_landmarks:
#         mpDraw.draw_landmarks(img, results.pose_landmarks, mpPose.POSE_CONNECTIONS)
#         for id, lm in enumerate(results.pose_landmarks.landmark):
#             h, w,c = img.shape
#             cx, cy = int(lm.x*w), int(lm.y*h)
#             cv2.circle(img, (cx, cy), 5, yellow, cv2.FILLED)


#     cTime = time.time()
#     fps = 1/(cTime-pTime)
#     pTime = cTime

#     cv2.putText(img, str(int(fps)), (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, light_green, 3)
#     angle_text_r = f"Right Bicep Angle: {str(round(r_bicep_angle, 2))}\nLeft Bicep Angle: {str(round(l_bicep_angle, 2))}"
#     cv2.putText(image, angle_text_string, (10, 30), font, 0.9, light_green, 2)

#     cv2.imshow("Image", img)
#     cv2.waitKey(1)